# 07-03 n8n 自动化流程

**n8n** 是开源的工作流自动化工具，擅长连接各种 API 和服务。

**本节目标**：n8n 核心概念、AI Agent 节点、与 LLM 集成

---

In [ ]:
print("""
n8n 核心概念:

1. Workflow（工作流）:
   - 触发器节点: Webhook / 定时 / 手动
   - 处理节点: HTTP请求 / Code / IF / Switch
   - AI 节点: LLM Chain / Agent / 向量检索
   - 输出节点: Slack / Email / 数据库写入

2. n8n AI 功能 (v1.19+):
   - AI Agent 节点: 内置 ReAct Agent
   - Tool 节点: 自定义工具（API调用、代码执行）
   - Vector Store 节点: Pinecone / Supabase 集成
   - Memory 节点: 对话记忆

3. B站广告场景:
   自动化监控: 定时查询广告数据 → CTR低于阈值 → AI分析原因 → 发送告警到飞书
   素材审核: Webhook接收新素材 → AI审核 → 人工确认 → 更新状态

4. n8n vs Dify vs Coze:
   n8n:  通用自动化 + AI，强在连接外部系统（500+集成）
   Dify: 纯 AI 应用，强在 RAG 和对话
   Coze: C端智能体，强在发布和插件生态
""")

In [ ]:
# n8n 是 Web 平台，展示 Webhook 触发 + AI 处理的代码等价实现
import json
from utils.llm_client import call_llm

def simulate_n8n_workflow(ad_data: dict) -> dict:
    """
    模拟 n8n 工作流: 广告数据监控 → AI分析 → 告警
    等价于 n8n 中: Webhook → IF → AI Agent → Slack
    """
    # Step 1: 触发条件检查 (n8n IF 节点)
    ctr = ad_data.get('clicks', 0) / max(ad_data.get('impressions', 1), 1) * 100
    if ctr >= 2.0:
        return {"action": "no_alert", "reason": f"CTR={ctr:.1f}% 正常"}
    
    # Step 2: AI 分析 (n8n AI Agent 节点)
    try:
        analysis = call_llm(
            f"广告数据: CTR={ctr:.1f}%, 点击{ad_data['clicks']}, 消耗{ad_data.get('cost',0)}元。分析原因并给建议(2句话)",
            system="你是广告优化AI", max_tokens=100
        )
    except Exception:
        analysis = f"CTR={ctr:.1f}%低于行业均值2.1%，建议优化创意素材和目标人群定向。"
    
    # Step 3: 发送告警 (n8n Slack/飞书节点)
    alert = {"action": "alert", "ctr": round(ctr, 2), "analysis": analysis, "channel": "#ad-alerts"}
    return alert

# 模拟运行
result = simulate_n8n_workflow({"impressions": 50000, "clicks": 500, "cost": 750})
print(f"工作流结果: {json.dumps(result, ensure_ascii=False, indent=2)}")

## 三平台对比总结

| 特性 | Dify | Coze | n8n |
|------|------|------|-----|
| 定位 | AI应用开发 | C端智能体 | 通用自动化 |
| 开源 | ✅ | ❌ | ✅ |
| RAG | ✅ 内置完善 | ✅ 知识库 | ✅ 向量检索节点 |
| Agent | ✅ | ✅ | ✅ AI Agent节点 |
| 外部集成 | API为主 | 插件市场 | 500+原生集成 |
| 部署 | Docker自部署 | 云端 | Docker自部署 |
| 最适合 | B端AI应用 | C端Bot | 系统间自动化 |

**Module 07 完成！** 下一步：`../08-openclaw/`